In [2]:
import pandas as pd
import numpy as np

# ============================================================
# CONFIG
# ============================================================

TRANSACTIONS_FILE = "LI-Small_Trans.csv"
ACCOUNTS_FILE = "LI-Small_accounts.csv"

OUTPUT_ACCOUNTS = "enriched_accounts.csv"
OUTPUT_TRANSACTIONS = "enriched_transactions.csv"

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)


# ============================================================
# YOUR DATA
# ============================================================

DISTRICT_WEIGHTS = {
    "Cyberabad Commissionerate": 2446265,
    "Hyderabad Commissionerate": 3943323,
    "Karimnagar Commissionerate": 1005711,
    "Khammam Commissionerate": 1401639,
    "Nizamabad Commissionerate": 1571022,
    "Rachakonda Commissionerate": 2446265,
    "Ramagundam Commissionerate": 795332,
    "Siddipet Commissionerate": 1012065,
    "Warangal Commissionerate": 1789395,
    "Adilabad": 708972,
    "Bhadradri Kothagudem": 1069261,
    "Jagityal": 985417,
    "Jayashankar Bhupalpalli": 416763,
    "Jogulamba Gadwal": 609990,
    "Kamareddy": 972625,
    "Kumaram Bheem Asifabad": 515812,
    "Mahabubabad": 774549,
    "Mahabubnagar": 919903,
    "Medak": 767428,
    "Nagarkurnool": 893308,
    "Nalgonda": 1618416,
    "Nirmal": 709418,
    "Rajanna Siricilla": 552037,
    "Sangareddy": 1527628,
    "Suryapet": 1099560,
    "Vikarabad": 927140,
    "Wanaparthy": 577758,
    "Railway Police Secunderabad": 3943323,
    "Mulug": 257744,
    "Narayanpet": 566874,
}

CRIME_COUNTS = {
    "Adilabad": 37,
    "Bhadradri Kothagudem": 117,
    "Cyberabad Commissionerate": 5424,
    "Hyderabad Commissionerate": 4436,
    "Jagityal": 156,
    "Jayashankar Bhupalpalli": 16,
    "Jogulamba Gadwal": 33,
    "Kamareddy": 295,
    "Karimnagar Commissionerate": 86,
    "Khammam Commissionerate": 237,
    "Kumaram Bheem Asifabad": 35,
    "Mahabubabad": 59,
    "Mahabubnagar": 87,
    "Medak": 127,
    "Mulug": 26,
    "Nagarkurnool": 76,
    "Nalgonda": 95,
    "Narayanpet": 49,
    "Nirmal": 37,
    "Nizamabad Commissionerate": 134,
    "Rachakonda Commissionerate": 2192,
    "Railway Police Secunderabad": 0,
    "Rajanna Siricilla": 63,
    "Ramagundam Commissionerate": 119,
    "Sangareddy": 286,
    "Siddipet Commissionerate": 246,
    "Suryapet": 119,
    "Vikarabad": 41,
    "Wanaparthy": 38,
    "Warangal Commissionerate": 631,
}

BANKS = [
    ("STATE BANK OF INDIA", 1215),
    ("UNION BANK OF INDIA", 683),
    ("HDFC BANK", 465),
    ("ICICI BANK", 398),
    ("CANARA BANK", 396),
    ("AXIS BANK", 199),
    ("BANK OF BARODA", 178),
    ("INDIAN BANK", 167),
    ("BANDHAN BANK", 159),
    ("PUNJAB NATIONAL BANK", 147),
    ("KOTAK MAHINDRA BANK", 123),
    ("INDIAN OVERSEAS BANK", 116),
    ("INDUSIND BANK", 107),
    ("CENTRAL BANK OF INDIA", 101),
    ("AU SMALL FIN.BANK", 92),
    ("BANK OF INDIA", 89),
    ("KARUR VYSYA BANK", 65),
    ("IDBI BANK", 55),
    ("IDFC FIRST BANK", 53),
    ("CITY UNION BANK", 47),
    ("SOUTH INDIAN BANK", 44),
    ("YES BANK", 43),
    ("FEDERAL BANK", 38),
    ("DCB BANK", 37),
    ("DBS BANK INDIA (E-LVB)", 35),
    ("CSB BANK LIMITED", 32),
    ("RBL BANK", 32),
    ("KARNATAKA BANK", 30),
    ("EQUITAS SMALL FIN. BANK", 28),
    ("INDIA POST PAYMENTS BANK", 23),
    ("PUNJAB AND SIND BANK", 17),
    ("KBS LOCAL AREA BANK", 14),
    ("TAMILNAD MERCANTILE BANK", 12),
]


# ============================================================
# LOAD DATA
# ============================================================

transactions = pd.read_csv(TRANSACTIONS_FILE)
accounts = pd.read_csv(ACCOUNTS_FILE)

# Handle duplicate "Account" columns produced by pandas
account_cols = [
    c for c in transactions.columns
    if c == "Account" or c.startswith("Account.")
]

if len(account_cols) >= 2:
    transactions = transactions.rename(columns={
        account_cols[0]: "From Account",
        account_cols[1]: "To Account"
    })

# Clean IDs
transactions["From Account"] = transactions["From Account"].astype(str).str.strip()
transactions["To Account"] = transactions["To Account"].astype(str).str.strip()
accounts["Account Number"] = accounts["Account Number"].astype(str).str.strip()

# Parse fields
transactions["Timestamp"] = pd.to_datetime(
    transactions["Timestamp"],
    format="%Y/%m/%d %H:%M",
    errors="coerce"
)

transactions["Amount Received"] = pd.to_numeric(
    transactions["Amount Received"], errors="coerce"
)

transactions["Amount Paid"] = pd.to_numeric(
    transactions["Amount Paid"], errors="coerce"
)

transactions["Is Laundering"] = (
    pd.to_numeric(
        transactions["Is Laundering"],
        errors="coerce"
    )
    .fillna(0)
    .astype(int)
    .astype(bool)
)




In [3]:
# ============================================================
# ACCOUNT-LEVEL BEHAVIOR
# ============================================================

outgoing = transactions.groupby("From Account").agg(
    outgoing_transactions=("From Account", "size"),
    outgoing_amount=("Amount Paid", "sum"),
    outgoing_laundering=("Is Laundering", "sum")
)

incoming = transactions.groupby("To Account").agg(
    incoming_transactions=("To Account", "size"),
    incoming_amount=("Amount Received", "sum"),
    incoming_laundering=("Is Laundering", "sum")
)

account_stats = outgoing.join(incoming, how="outer").fillna(0)

account_stats["total_transactions"] = (
    account_stats["outgoing_transactions"]
    + account_stats["incoming_transactions"]
)

account_stats["total_amount"] = (
    account_stats["outgoing_amount"]
    + account_stats["incoming_amount"]
)

account_stats["laundering_transactions"] = (
    account_stats["outgoing_laundering"]
    + account_stats["incoming_laundering"]
)

account_stats["laundering_rate"] = np.where(
    account_stats["total_transactions"] > 0,
    account_stats["laundering_transactions"]
    / account_stats["total_transactions"],
    0
)

account_stats = account_stats.reset_index()
account_stats = account_stats.rename(
    columns={"index": "Account Number"}
)




In [4]:
# ============================================================
# DISTRICT TARGETS
# ============================================================

district_df = pd.DataFrame({
    "District": list(DISTRICT_WEIGHTS.keys()),
    "Population": list(DISTRICT_WEIGHTS.values())
})

district_df["Crime Count"] = (
    district_df["District"]
    .map(CRIME_COUNTS)
    .fillna(0)
)

district_df["Population Share"] = (
    district_df["Population"]
    / district_df["Population"].sum()
)

district_df["Crime Share"] = (
    district_df["Crime Count"]
    / district_df["Crime Count"].sum()
)

# Crime intensity relative to population.
# Used to bias suspicious accounts toward higher-crime districts.
district_df["Crime_Population_Index"] = (
    district_df["Crime Share"]
    / district_df["Population Share"]
)

# Avoid zero/invalid values
district_df["Crime_Population_Index"] = (
    district_df["Crime_Population_Index"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)




In [5]:
# ============================================================
# ACCOUNT RISK SCORE
# ============================================================

account_df = account_stats.copy()

# Log prevents a handful of huge accounts from dominating.
account_df["risk_score"] = (
    np.log1p(account_df["laundering_transactions"])
    * (1 + account_df["laundering_rate"])
)

# High-risk accounts first
account_df = account_df.sort_values(
    "risk_score",
    ascending=False
).reset_index(drop=True)


# ============================================================
# TARGET NUMBER OF ACCOUNTS PER DISTRICT
# ============================================================

n_accounts = len(account_df)

district_df["Target Accounts"] = (
    district_df["Population Share"] * n_accounts
).round().astype(int)

# Correct rounding error
difference = (
    n_accounts
    - district_df["Target Accounts"].sum()
)

if difference != 0:
    idx = district_df["Population Share"].idxmax()
    district_df.loc[idx, "Target Accounts"] += difference




In [10]:
# ============================================================
# DISTRICT ASSIGNMENT
# ============================================================

capacity = dict(
    zip(
        district_df["District"],
        district_df["Target Accounts"]
    )
)

risk_multiplier = dict(
    zip(
        district_df["District"],
        district_df["Crime_Population_Index"]
    )
)

districts = list(capacity.keys())

# Normalize risk multiplier
mean_multiplier = np.mean(
    [v for v in risk_multiplier.values() if v > 0]
)

if mean_multiplier > 0:
    for d in risk_multiplier:
        if risk_multiplier[d] > 0:
            risk_multiplier[d] /= mean_multiplier

print("Accounts: ", len(account_df))
assigned_districts = []
# print("Accounts: ")
for _, account in account_df.iterrows():

    available = [
        d for d in districts
        if capacity[d] > 0
    ]
    if _ % 100 == 0:
        print("Row: ", _)
        print("Available districts:", available)
    # If no capacity remains, stop
    if not available:
        break

    # Base probability from population distribution
    # plus crime/population calibration.
    weights = np.array([
        risk_multiplier[d]
        for d in available
    ], dtype=float)

    # Give every district a non-zero probability.
    weights = np.maximum(weights, 0.05)

    # High-risk accounts get stronger geographic calibration.
    risk = account["risk_score"]

    # Saturating function:
    # low-risk accounts ≈ population-driven
    # high-risk accounts ≈ crime-driven
    alpha = min(
        0.75,
        risk / (risk + 3)
    )

    # Population probability
    pop_weights = np.array([
        district_df.loc[
            district_df["District"] == d,
            "Population Share"
        ].iloc[0]
        for d in available
    ])

    pop_weights = pop_weights / pop_weights.sum()

    # Crime-intensity probability
    crime_weights = weights / weights.sum()

    final_weights = (
        (1 - alpha) * pop_weights
        + alpha * crime_weights
    )

    final_weights = final_weights / final_weights.sum()

    chosen = rng.choice(
        available,
        p=final_weights
    )

    assigned_districts.append(chosen)
    capacity[chosen] -= 1


account_df["District"] = assigned_districts




Accounts:  705903
Row:  0
Available districts: ['Cyberabad Commissionerate', 'Hyderabad Commissionerate', 'Karimnagar Commissionerate', 'Khammam Commissionerate', 'Nizamabad Commissionerate', 'Rachakonda Commissionerate', 'Ramagundam Commissionerate', 'Siddipet Commissionerate', 'Warangal Commissionerate', 'Adilabad', 'Bhadradri Kothagudem', 'Jagityal', 'Jayashankar Bhupalpalli', 'Jogulamba Gadwal', 'Kamareddy', 'Kumaram Bheem Asifabad', 'Mahabubabad', 'Mahabubnagar', 'Medak', 'Nagarkurnool', 'Nalgonda', 'Nirmal', 'Rajanna Siricilla', 'Sangareddy', 'Suryapet', 'Vikarabad', 'Wanaparthy', 'Railway Police Secunderabad', 'Mulug', 'Narayanpet']
Row:  100
Available districts: ['Cyberabad Commissionerate', 'Hyderabad Commissionerate', 'Karimnagar Commissionerate', 'Khammam Commissionerate', 'Nizamabad Commissionerate', 'Rachakonda Commissionerate', 'Ramagundam Commissionerate', 'Siddipet Commissionerate', 'Warangal Commissionerate', 'Adilabad', 'Bhadradri Kothagudem', 'Jagityal', 'Jayashankar

KeyboardInterrupt: 

In [ ]:
# ============================================================
# BANK ASSIGNMENT
# ============================================================

bank_df = pd.DataFrame(
    BANKS,
    columns=["Indian Bank", "Branch Weight"]
)

bank_df["Bank Share"] = (
    bank_df["Branch Weight"]
    / bank_df["Branch Weight"].sum()
)

account_df["Indian Bank"] = rng.choice(
    bank_df["Indian Bank"].values,
    size=len(account_df),
    p=bank_df["Bank Share"].values
)


# ============================================================
# ADD DISTRICT / BANK INFORMATION TO ACCOUNT DATASET
# ============================================================

account_enrichment = account_df[
    [
        "Account Number",
        "District",
        "Indian Bank",
        "total_transactions",
        "total_amount",
        "laundering_transactions",
        "laundering_rate"
    ]
].copy()

account_enrichment = account_enrichment.merge(
    district_df[
        [
            "District",
            "Population",
            "Crime Count",
            "Population Share",
            "Crime Share",
            "Crime_Population_Index"
        ]
    ],
    on="District",
    how="left"
)

account_enrichment = account_enrichment.merge(
    bank_df[
        [
            "Indian Bank",
            "Branch Weight",
            "Bank Share"
        ]
    ],
    on="Indian Bank",
    how="left"
)

enriched_accounts = accounts.merge(
    account_enrichment,
    on="Account Number",
    how="left"
)


# ============================================================
# PROPAGATE ACCOUNT LOCATION INTO TRANSACTIONS
# ============================================================

lookup = account_enrichment.set_index("Account Number")

transactions["From District"] = (
    transactions["From Account"]
    .map(lookup["District"])
)

transactions["To District"] = (
    transactions["To Account"]
    .map(lookup["District"])
)

transactions["From Indian Bank"] = (
    transactions["From Account"]
    .map(lookup["Indian Bank"])
)

transactions["To Indian Bank"] = (
    transactions["To Account"]
    .map(lookup["Indian Bank"])
)




In [ ]:
# ============================================================
# VALIDATION
# ============================================================

validation = (
    account_df
    .groupby("District")
    .agg(
        Accounts=("Account Number", "count"),
        Transactions=("total_transactions", "sum"),
        Laundering_Transactions=(
            "laundering_transactions",
            "sum"
        )
    )
    .reset_index()
    .merge(
        district_df[
            [
                "District",
                "Population Share",
                "Crime Share",
                "Population",
                "Crime Count"
            ]
        ],
        on="District",
        how="left"
    )
)

validation["Actual Account Share"] = (
    validation["Accounts"]
    / validation["Accounts"].sum()
)

validation["Actual Laundering Share"] = (
    validation["Laundering_Transactions"]
    / validation["Laundering_Transactions"].sum()
)

validation["Account Share Error"] = (
    validation["Actual Account Share"]
    - validation["Population Share"]
)

validation["Laundering Share Error"] = (
    validation["Actual Laundering Share"]
    - validation["Crime Share"]
)

validation["Laundering Rate"] = np.where(
    validation["Transactions"] > 0,
    validation["Laundering_Transactions"]
    / validation["Transactions"],
    0
)


# ============================================================
# SAVE
# ============================================================

enriched_accounts.to_csv(
    OUTPUT_ACCOUNTS,
    index=False
)

transactions.to_csv(
    OUTPUT_TRANSACTIONS,
    index=False
)


# ============================================================
# REPORT
# ============================================================

print("=" * 70)
print("IBM DATASET ENRICHMENT COMPLETE")
print("=" * 70)

print(f"\nOriginal transactions : {len(transactions):,}")
print(f"Original accounts     : {len(accounts):,}")

print(f"\nSaved:")
print(f"  {OUTPUT_ACCOUNTS}")
print(f"  {OUTPUT_TRANSACTIONS}")

print("\nMissing district assignments:")
print(
    "  Accounts:",
    enriched_accounts["District"].isna().sum()
)

print(
    "  From transactions:",
    transactions["From District"].isna().sum()
)

print(
    "  To transactions:",
    transactions["To District"].isna().sum()
)

print("\nDistrict validation:")
display(
    validation[
        [
            "District",
            "Population Share",
            "Actual Account Share",
            "Crime Share",
            "Actual Laundering Share",
            "Laundering Rate"
        ]
    ].sort_values(
        "Crime Share",
        ascending=False
    )
)

print("\nMean absolute account-share error:",
      f"{validation['Account Share Error'].abs().mean():.4%}")

print("Mean absolute laundering-share error:",
      f"{validation['Laundering Share Error'].abs().mean():.4%}")

print("\nBank distribution:")
display(
    account_df["Indian Bank"]
    .value_counts()
    .rename("Accounts")
    .to_frame()
    .assign(
        Actual_Share=lambda x:
            x["Accounts"] / x["Accounts"].sum()
    )
)

print("\nSample enriched transactions:")
display(transactions.head(10))

print("\nSample enriched accounts:")
display(enriched_accounts.head(10))